In [ ]:
# SkinNova — Gemma 4 E2B LoRA (plan §8, test.md §5 L1–L4). Generated by ml/llm/notebooks/make_kaggle.py — edit there.
MODE = "smoke"


In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install -q "huggingface_hub>=1.5.0,<2.0" torchcodec
!pip install -q --no-deps --upgrade timm


In [ ]:
import glob, json, os, random, sys, time, re, math
import torch
torch._dynamo.config.recompile_limit = 64
DATA = os.path.dirname(glob.glob("/kaggle/input/**/sft_train.jsonl", recursive=True)[0])
sys.path.insert(0, DATA + "/code")
from ml.llm.validate import validate, guard_text, count_sentences, load_rx_terms   # same validator as the app (OutputParser.kt)
from ml.voice.intake import validate_intake
OUT = "/kaggle/working"; os.makedirs(f"{OUT}/reports", exist_ok=True)
LABELS = json.load(open(f"{DATA}/code/labels.json")); KEYS = [c["key"] for c in LABELS["classes"]]
FLOOR = {c["key"]: c["tier_floor"] for c in LABELS["classes"]}
RX = [l.strip().lower() for l in open(f"{DATA}/code/rx_terms.txt") if l.strip() and not l.startswith("#")]
TR = {t: i for i, t in enumerate(["LOW", "MODERATE", "HIGH", "URGENT"])}
def load(n): return [json.loads(l) for l in open(f"{DATA}/{n}.jsonl")]
train, sval, lval = load("sft_train"), load("sft_val"), load("llm_val")
general = load("general_regression")
print("MODE", MODE, "| train", len(train), "sft_val", len(sval), "llm_val", len(lval), "| GPU", torch.cuda.get_device_name(0))
REPORT = {"mode": MODE, "gpu": torch.cuda.get_device_name(0), "dataset_meta": json.load(open(f"{DATA}/sft_data.json"))}
def save_report():
    json.dump(REPORT, open(f"{OUT}/reports/{MODE}.json", "w"), indent=1, default=str)


In [ ]:
from PIL import Image
def to_conv(r, with_answer=True, image_override=None):
    msgs = []
    for m in r["messages"]:
        if m["role"] == "system":
            msgs.append({"role": "system", "content": m["content"]}); continue   # gemma-4 template trims a *string* system turn
        if m["role"] == "assistant" and not with_answer:
            continue
        content = []
        for c in m["content"]:
            if c["type"] == "image":
                img = image_override if image_override is not None else Image.open(f'{DATA}/{r["image"]}').convert("RGB")
                content.append({"type": "image", "image": img})
            else:
                content.append({"type": "text", "text": c["text"]})
        msgs.append({"role": m["role"], "content": content})
    return {"messages": msgs}
def target(r): return r["messages"][-1]["content"][0]["text"]


In [ ]:
from unsloth import FastVisionModel, get_chat_template
t0 = time.time()
try:
    model, processor = FastVisionModel.from_pretrained("unsloth/gemma-4-E2B-it", load_in_4bit=False, use_gradient_checkpointing="unsloth")
    REPORT["load_in_4bit"] = False
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    model, processor = FastVisionModel.from_pretrained("unsloth/gemma-4-E2B-it", load_in_4bit=True, use_gradient_checkpointing="unsloth")
    REPORT["load_in_4bit"] = True
processor = get_chat_template(processor, "gemma-4")
model = FastVisionModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    random_state=3407, use_rslora=False, loftq_config=None, target_modules="all-linear")
REPORT["load_s"] = round(time.time() - t0, 1)
REPORT["trainable_params"] = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(REPORT)


In [ ]:
# ---- L1 pre-training checks (must print PASS) ----
from unsloth.trainer import UnslothVisionDataCollator
collator = UnslothVisionDataCollator(model, processor, train_on_responses_only=True,
                                     instruction_part="<|turn>user\n", response_part="<|turn>model\n", max_seq_length=2048)
s = to_conv(next(r for r in train if r["task"] == "T1"))
txt = processor.apply_chat_template(s["messages"], tokenize=False, add_generation_prompt=False)
assert txt.count("<bos>") <= 1 and "<|turn>model" in txt and "<|turn>system" in txt, txt[:400]
assert txt.index("<|image|>") < txt.index("IMAGE_MODEL_TOP3"), "image must come before text"
batch = collator([s]); lab = batch["labels"][0]; ids = batch["input_ids"][0]
visible = processor.tokenizer.decode(ids[lab != -100])
assert visible.strip().startswith("{") and "IMAGE_MODEL_TOP3" not in visible, visible[:300]
lens = []
for r in random.Random(0).sample(train, 200):
    b = collator([to_conv(r)]); lens.append(int(b["input_ids"].shape[1]))
lens.sort(); p99 = lens[int(.99 * len(lens)) - 1]
REPORT["L1"] = {"template_ok": True, "masking_ok": True, "visible_head": visible[:120], "tokens_p50": lens[100], "tokens_p99": p99, "tokens_max": lens[-1]}
assert p99 <= 2048, p99
print("L1 PASS", REPORT["L1"]); save_report()


In [ ]:
# ---- inference helpers ----
def gen(r, max_new_tokens=700, image_override=None, adapter=True):
    conv = to_conv(r, with_answer=False, image_override=image_override)["messages"]
    imgs = [c["image"] for m in conv if isinstance(m["content"], list) for c in m["content"] if c["type"] == "image"]
    tmpl = [{"role": m["role"], "content": m["content"] if isinstance(m["content"], str) else
             [{"type": "image"} if c["type"] == "image" else {"type": "text", "text": c["text"]} for c in m["content"]]} for m in conv]
    text = processor.apply_chat_template(tmpl, add_generation_prompt=True)
    inputs = (processor(imgs[0], text, add_special_tokens=False, return_tensors="pt") if imgs else
              processor(text=text, add_special_tokens=False, return_tensors="pt")).to("cuda")
    def _g():
        return model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    if adapter:
        out = _g()
    else:
        with model.disable_adapter():
            out = _g()
    return processor.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

from ml.eval.llm_metrics import score_analysis as _sa, score_extract as _se, summarize, summarize_extract, ANALYSIS_KEYS
def score_analysis(r, text): return _sa(r["meta"], text)
def score_extract(r, text): return _se(r["meta"], text)


In [ ]:
# ---- base-model sanity (E0) on a few llm_val cases: proves the eval path works before any training ----
FastVisionModel.for_inference(model)
t0 = time.time(); base = []
for r in [x for x in lval if x["task"] == "T1"][:3]:
    out = gen(r, 400, adapter=False); base.append({"id": r["id"], "out": out[:600], **score_analysis(r, out)})
REPORT["E0_sanity"] = {"cases": base, "s_per_case": round((time.time() - t0) / 3, 1)}
print(json.dumps(REPORT["E0_sanity"], indent=1)[:3000]); save_report()


In [ ]:
from trl import SFTTrainer, SFTConfig
def sft_args(**kw):
    base = dict(per_device_train_batch_size=1, gradient_accumulation_steps=8, max_grad_norm=0.3, warmup_ratio=0.03,
                learning_rate=2e-4, logging_steps=5, optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="cosine",
                seed=3407, output_dir=f"{OUT}/out", report_to="none", remove_unused_columns=False, dataset_text_field="",
                dataset_kwargs={"skip_prepare_dataset": True}, max_length=2048, save_strategy="no")
    base.update(kw); return SFTConfig(**base)

if MODE == "smoke":
    # ---- L2 overfit test: 16 records, 100 steps → loss < 0.3 and ≥ 14/16 exact greedy reproductions ----
    small = [r for r in train if r["task"] in {"T1", "T2", "T3", "T4", "T9"}][:12] + [r for r in train if r["task"] in {"T6", "T7", "T8"}][:4]
    FastVisionModel.for_training(model)
    trainer = SFTTrainer(model=model, train_dataset=[to_conv(r) for r in small], processing_class=processor.tokenizer,
                         data_collator=collator, args=sft_args(gradient_accumulation_steps=1, max_steps=100, warmup_ratio=0.0))
    t0 = time.time(); st = trainer.train(); dt = time.time() - t0
    losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    FastVisionModel.for_inference(model)
    exact = 0; outs = []
    for r in small:
        o = gen(r, 700).strip(); ok = o == target(r).strip(); exact += ok; outs.append({"id": r["id"], "exact": ok, "out": o[:300]})
    REPORT["L2"] = {"first_loss": losses[0], "final_loss": losses[-1], "losses": losses, "exact": exact, "n": len(small),
                    "pass": bool(losses[-1] < 0.3 and exact >= 14), "train_s": round(dt, 1), "s_per_step_bs1": round(dt / 100, 2),
                    "peak_mem_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2), "outs": outs}
    steps_full = math.ceil(len(train) / 8) * 2
    REPORT["projection"] = {"full_steps": steps_full, "est_full_train_h": round(steps_full * 8 * dt / 100 / 3600, 2)}
    print("L2", "PASS" if REPORT["L2"]["pass"] else "FAIL", {k: REPORT["L2"][k] for k in ["first_loss", "final_loss", "exact", "train_s", "peak_mem_gb"]},
          REPORT["projection"])
    save_report()


In [ ]:
if MODE == "full":
    # ---- E1: full training (plan §8.3), eval loss on sft_val ----
    FastVisionModel.for_training(model)
    trainer = SFTTrainer(model=model, train_dataset=[to_conv(r) for r in train], eval_dataset=[to_conv(r) for r in sval],
                         processing_class=processor.tokenizer, data_collator=collator,
                         args=sft_args(num_train_epochs=2, eval_strategy="steps", eval_steps=100, per_device_eval_batch_size=1,
                                       save_strategy="steps", save_steps=100, save_total_limit=2, load_best_model_at_end=True,
                                       metric_for_best_model="eval_loss", greater_is_better=False))
    t0 = time.time(); trainer.train(); dt = time.time() - t0
    REPORT["train"] = {"wall_h": round(dt / 3600, 2), "log": trainer.state.log_history, "best": trainer.state.best_metric,
                       "peak_mem_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2)}
    model.save_pretrained(f"{OUT}/lora"); processor.save_pretrained(f"{OUT}/lora")
    save_report()


In [ ]:
if MODE == "full":
    # ---- L4: E0 (adapter off) vs E1 on llm_val; image ablation; extraction; forgetting set ----
    FastVisionModel.for_inference(model)
    an = [r for r in lval if r["task"] in {"T1", "T9"}][:120]
    ex = [r for r in lval if r["task"] == "T6"]
    res = {"E0": [], "E1": []}
    for arm, adapter in [("E1", True), ("E0", False)]:
        for r in an:
            o = gen(r, 700, adapter=adapter); res[arm].append({"id": r["id"], "task": r["task"], **score_analysis(r, o), "out": o[:1500]})
        save_report()
    REPORT["L4_analysis"] = {arm: summarize(rows, ANALYSIS_KEYS) for arm, rows in res.items()}
    REPORT["L4_analysis_rows"] = res
    gray = Image.new("RGB", (512, 512), (128, 128, 128)); diff = 0; abl = []
    for r, row in list(zip(an, res["E1"]))[:30]:
        o = gen(r, 700, image_override=gray); s = score_analysis(r, o); changed = set(s["cats"]) != set(row["cats"]); diff += changed
        abl.append({"id": r["id"], "real": row["cats"], "gray": s["cats"], "changed": changed})
    REPORT["image_ablation"] = {"changed_rate": round(diff / 30, 3), "rows": abl}
    xr = {"E1": [], "E0": []}
    for arm, adapter in [("E1", True), ("E0", False)]:
        for r in ex:
            o = gen(r, 300, adapter=adapter); xr[arm].append({"id": r["id"], **score_extract(r, o), "out": o[:600]})
    REPORT["L4_extract"] = {arm: summarize_extract(rows) for arm, rows in xr.items()}
    gr = []
    for q in general:
        r = {"messages": [{"role": "user", "content": [{"type": "text", "text": q["prompt"]}]}]}
        gr.append({"id": q["id"], "prompt": q["prompt"], "E1": gen(r, 150), "E0": gen(r, 150, adapter=False)})
    REPORT["general_regression"] = gr
    save_report()
    print(json.dumps({k: REPORT[k] for k in ["L4_analysis", "L4_extract"]}, indent=1), REPORT["image_ablation"]["changed_rate"])


In [ ]:
if MODE == "full":
    # merged 16-bit weights for the .litertlm export notebook (test L5 merge parity runs there)
    model.save_pretrained_merged(f"{OUT}/merged", processor, save_method="merged_16bit")
    print(os.listdir(f"{OUT}/merged"))
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN"); user = UserSecretsClient().get_secret("HF_USER")
        model.push_to_hub(f"{user}/skinnova-e2b-lora-v1", token=tok, private=True); processor.push_to_hub(f"{user}/skinnova-e2b-lora-v1", token=tok, private=True)
        REPORT["hf_lora_repo"] = f"{user}/skinnova-e2b-lora-v1"
    except Exception as e:
        REPORT["hf_push"] = f"skipped: {type(e).__name__}"
    save_report()
